In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
import warnings
import lightgbm as lgb
warnings.filterwarnings("ignore")

# Load data
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")

X = train.drop(["Churn", "id"], axis=1)
y = train["Churn"].map({"Yes": 1, "No": 0})

test_ids = test["id"]
X_test = test.drop(["id"], axis=1)

# Now identify column types (after feature engineering)
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"\nTarget distribution:\n{y.value_counts(normalize=True)}")

# Create preprocessor
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ("num", "passthrough", numeric_cols)
])

# Create models with best parameters
xgb_best = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=700,
        max_depth=5,
        learning_rate=0.07,
        subsample=0.9,
        min_child_weight=5,
        colsample_bytree=1.0,
        random_state=42,
        eval_metric="auc"
    ))
])

lgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", lgb.LGBMClassifier(
        n_estimators=700,
        max_depth=5,
        learning_rate=0.07,
        subsample=0.9,
        min_child_weight=5,
        colsample_bytree=1.0,
        random_state=42,
        verbose=-1
    ))
])

# Split data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train models
print("\nTraining XGBoost...")
xgb_best.fit(X_train, y_train)
print("Training LightGBM...")
lgb_model.fit(X_train, y_train)

# Get validation probabilities
xgb_val = xgb_best.predict_proba(X_val)[:, 1]
lgb_val = lgb_model.predict_proba(X_val)[:, 1]

# Find best ensemble weight
print("\nEnsemble validation results:")
best_weight = 0.5
best_auc = 0
for weight_xgb in [0.5, 0.6, 0.7]:
    ensemble_val = weight_xgb * xgb_val + (1 - weight_xgb) * lgb_val
    auc = roc_auc_score(y_val, ensemble_val)
    print(f"Weight XGB={weight_xgb}: Ensemble AUC = {auc:.4f}")
    if auc > best_auc:
        best_auc = auc
        best_weight = weight_xgb

print(f"\nBest ensemble weight: {best_weight} with AUC: {best_auc:.4f}")

# Get test predictions
xgb_test = xgb_best.predict_proba(X_test)[:, 1]
lgb_test = lgb_model.predict_proba(X_test)[:, 1]

# Create ensemble predictions
test_proba = best_weight * xgb_test + (1 - best_weight) * lgb_test

# Create submission
submission = pd.DataFrame({
    "id": test_ids,
    "Churn": test_proba
})

submission.to_csv("submission.csv", index=False)
print("\nSubmission file created: submission.csv")
print(submission.head())

# Optional: Cross-validation on full dataset
print("\nPerforming cross-validation...")
cv_scores_xgb = cross_val_score(xgb_best, X, y, cv=5, scoring="roc_auc", n_jobs=-1)
print(f"XGBoost CV AUC: {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std():.4f})")

cv_scores_lgb = cross_val_score(lgb_model, X, y, cv=5, scoring="roc_auc", n_jobs=-1)
print(f"LightGBM CV AUC: {cv_scores_lgb.mean():.4f} (+/- {cv_scores_lgb.std():.4f})")

Categorical columns (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numeric columns (4): ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Target distribution:
Churn
0    0.774792
1    0.225208
Name: proportion, dtype: float64

Training XGBoost...
Training LightGBM...

Ensemble validation results:
Weight XGB=0.5: Ensemble AUC = 0.9172
Weight XGB=0.6: Ensemble AUC = 0.9173
Weight XGB=0.7: Ensemble AUC = 0.9172

Best ensemble weight: 0.6 with AUC: 0.9173

Submission file created: submission.csv
       id     Churn
0  594194  0.076931
1  594195  0.000494
2  594196  0.110531
3  594197  0.003750
4  594198  0.524211

Performing cross-validation...
XGBoost CV AUC: 0.9165 (+/- 0.0010)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM CV AUC: 0.9163 (+/- 0.0010)
